<a href="https://colab.research.google.com/github/MELES-DS/DATA-SCIENCE-CODES/blob/main/updated_attenndance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import io
import os


# =====================================================
# GLOBAL STORAGE
# =====================================================

datasets = {}
dataset_months = {}
sorted_datasets = {}



# =====================================================
# MONTH ORDER
# =====================================================

month_order = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}



# =====================================================
# DETECT MONTH FROM COLUMN HEADERS
# Example:
# July 1, July 2, July 3 ...
# =====================================================

def detect_month_from_dataset(df):

    month_count = {
        month: 0
        for month in month_order
    }


    for column in df.columns:

        column_name = str(column).strip().lower()


        for month in month_order:

            if month.lower() in column_name:

                month_count[month] += 1



    detected_month = max(
        month_count,
        key=month_count.get
    )


    if month_count[detected_month] > 0:

        return detected_month


    return "Unknown"





# =====================================================
# UPLOAD DATASET
# =====================================================

upload = widgets.FileUpload(
    accept=".xlsx,.xls,.xlsm,.xlsb,.csv",
    multiple=True,
    description="📂 Upload Files"
)


display(upload)





# =====================================================
# LOAD FILES
# =====================================================

def load_file(change):

    global datasets
    global dataset_months
    global sorted_datasets


    datasets = {}
    dataset_months = {}
    sorted_datasets = {}



    for key, file_dict in upload.value.items():


        filename = file_dict["metadata"]["name"]


        extension = os.path.splitext(filename)[1].lower()


        file_data = io.BytesIO(
            file_dict["content"]
        )


        try:


            if extension == ".csv":

                df = pd.read_csv(file_data)



            elif extension in [
                ".xlsx",
                ".xls",
                ".xlsm",
                ".xlsb"
            ]:

                df = pd.read_excel(
                    file_data,
                    header=1
                )


            else:

                print(
                    "❌ Unsupported file:",
                    filename
                )

                continue



            # Lowercase column names

            df.columns = (
                df.columns
                .astype(str)
                .str.strip()
                .str.lower()
            )



            datasets[filename] = df



            dataset_months[filename] = (
                detect_month_from_dataset(df)
            )



            print(
                "✅ Loaded:",
                filename,
                "| Month:",
                dataset_months[filename],
                "| Shape:",
                df.shape
            )



        except Exception as e:

            print(
                "❌ Error loading",
                filename,
                ":",
                e
            )



    # Automatic chronological sorting

    sorted_datasets = dict(
        sorted(
            datasets.items(),
            key=lambda x:
            month_order.get(
                dataset_months[x[0]],
                99
            )
        )
    )



    print("\n📅 Automatic chronological order:")


    for filename in sorted_datasets:

        print(
            dataset_months[filename],
            "→",
            filename
        )



upload.observe(
    load_file,
    names="value"
)





# =====================================================
# SELECT MONTH BUTTON
# =====================================================

select_month_button = widgets.Button(
    description="📅 Select Month",
    button_style="info"
)


display(select_month_button)





# =====================================================
# MONTH CHECKBOXES
# =====================================================

month_boxes = {}


month_box_container = widgets.VBox([])



for month in month_order:

    box = widgets.Checkbox(
        value=False,
        description=month
    )

    month_boxes[month] = box



month_box_container.children = list(
    month_boxes.values()
)



month_box_container.layout.display = "none"


display(month_box_container)





# =====================================================
# DISPLAY DATASET BUTTON
# =====================================================

display_button = widgets.Button(
    description="▶ Display Dataset",
    button_style="success"
)


display(display_button)


output = widgets.Output()

display(output)





# =====================================================
# SHOW / HIDE MONTH CHECKBOX
# =====================================================

def show_month_selector(button):

    if month_box_container.layout.display == "none":

        month_box_container.layout.display = "block"

        select_month_button.description = (
            "❌ Hide Month"
        )


    else:

        month_box_container.layout.display = "none"

        select_month_button.description = (
            "📅 Select Month"
        )



select_month_button.on_click(
    show_month_selector
)





# =====================================================
# DISPLAY FUNCTION
# =====================================================

def display_selected(button):

    with output:

        clear_output()



        if not datasets:

            print(
                "⚠ Please upload dataset files first."
            )

            return



        # Selected months in checkbox order

        selected_months = [

            month

            for month, box in month_boxes.items()

            if box.value

        ]



        # -------------------------------------------------
        # No checkbox selected:
        # Use chronological order
        # -------------------------------------------------

        if not selected_months:

            display_order = sorted_datasets



        # -------------------------------------------------
        # Checkbox selected:
        # Use selected month order
        # -------------------------------------------------

        else:


            display_order = {}

            missing_months = []



            for selected_month in selected_months:


                found = False



                for filename, df in sorted_datasets.items():


                    if dataset_months[filename] == selected_month:


                        display_order[filename] = df

                        found = True

                        break



                if not found:

                    missing_months.append(
                        selected_month
                    )



            if missing_months:


                print(
                    "⚠ Select the correct month according to your uploaded file."
                )


                print(
                    "Missing:",
                    ", ".join(missing_months)
                )


                return





        # -------------------------------------------------
        # Display result
        # -------------------------------------------------

        print(
            "📅 Dataset display order:"
        )


        for filename, df in display_order.items():


            print("=" * 80)


            print(
                "📅 Month:",
                dataset_months[filename]
            )


            print(
                "📂 File:",
                filename
            )


            print(
                "📊 Shape:",
                df.shape
            )


            print("=" * 80)


            display(
                df.head()
            )



display_button.on_click(
    display_selected
)

FileUpload(value={}, accept='.xlsx,.xls,.xlsm,.xlsb,.csv', description='📂 Upload Files', multiple=True)

Button(button_style='info', description='📅 Select Month', style=ButtonStyle())

Button(button_style='success', description='▶ Display Dataset', style=ButtonStyle())

Output()

✅ Loaded: DIY Attendance preparation  (1).xlsx | Month: July | Shape: (858, 36)
✅ Loaded: DIY Attendance preparation .xlsx | Month: July | Shape: (858, 36)

📅 Automatic chronological order:
July → DIY Attendance preparation  (1).xlsx
July → DIY Attendance preparation .xlsx


In [ ]:
print(datasets.keys())

dict_keys(['DIY Attendance preparation  (1).xlsx', 'DIY Attendance preparation .xlsx'])


In [ ]:

print(df.columns)

NameError: name 'df' is not defined

In [ ]:

for name, data in datasets.items():
    print(f"✅ Loaded: {filename}")
    print(f"📊 Shape: {df.shape}")
    print(name)
    display(data.head())
    print("\n \n \n")

NameError: name 'filename' is not defined